# Transformers and BERT exploration

### 0. System and frameworks check.

In [1]:
import transformers

print(transformers.__version__)

5.17.0


### 1. BERT Tokenizer

In [2]:
# importing the essential 
from transformers import AutoTokenizer

# telling the transformers lib to load the best tokenizer for pre-trained bert model 
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [3]:
# starting with some small text 
text = "The patient has hypertension and chest pain."

tokens = tokenizer.tokenize(text)

print(tokens)

['the', 'patient', 'has', 'hyper', '##tension', 'and', 'chest', 'pain', '.']


The word "hypertension" is being splitted because of the BERT technique called WordPiece tokenization. 

Since that word isn't contained into its vocabulary, it divides it in two sub-parts. 

The "##" sign means that the piece who contains it has to be attached to the previous one.

In [4]:
# going towards numerical translation of tokens
encoded = tokenizer(text, return_tensors="pt") # return as pytorch tensors instead of python list

print(encoded)

{'input_ids': tensor([[  101,  1996,  5776,  2038, 23760, 29048,  1998,  3108,  3255,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


Where:

101 = [CLS] - start of sequence

102 = [SEP] - end of sequence

token_type_ids = to distinguish two different sequences

attention_mask = 1 / 0 boolean

with 1 = real token 

0 = padding

### 2. BERT Model

In [5]:
from transformers import AutoModel 

bert = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
outputs = bert(**encoded)

print(outputs.last_hidden_state.shape)

torch.Size([1, 11, 768])


as [batch_size, sequence_length, hidden_size]

with hidden_size = vector dimension of each token

In [7]:
# first token of sequence
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(cls_embedding.shape)

torch.Size([1, 768])


In [8]:
print(cls_embedding)    # actual vector

tensor([[-4.4714e-01,  3.5689e-01, -1.1640e-01, -5.5167e-01,  2.4613e-02,
          1.6070e-01,  2.2225e-01,  2.8122e-01, -2.7628e-01, -3.7582e-01,
         -6.5440e-01, -2.5021e-01,  1.3723e-01,  2.3699e-01,  4.8735e-02,
          1.0158e-01, -2.5534e-01,  1.8926e-01, -2.0275e-01, -7.9744e-02,
         -3.4659e-02, -4.5726e-01,  2.0427e-01,  1.3796e-02,  8.9171e-02,
         -3.4374e-01, -1.7792e-01, -7.6753e-02, -1.5578e-01, -5.7843e-01,
         -2.5256e-01, -3.6601e-01, -5.1925e-01, -6.1502e-01,  1.7284e-01,
          1.6684e-01,  2.2257e-01, -1.1867e-02,  1.1652e-01,  1.2188e-01,
         -4.4463e-01, -2.3525e-01,  5.5458e-01,  2.9570e-01,  2.0710e-01,
         -1.0045e-01, -2.3859e+00, -2.1394e-01,  3.4492e-01, -4.0423e-01,
          4.1643e-01,  1.2062e-01,  4.9736e-01,  5.6682e-01,  1.3647e-01,
          5.5068e-01, -7.5438e-01,  3.6723e-01,  1.9767e-01,  1.1513e-01,
          4.1459e-01,  1.6426e-01, -1.1051e-02, -4.0379e-01, -2.6723e-01,
          2.8587e-01,  5.0260e-02,  3.

Path:

abstract --> tokenizer --> BERT --> [CLS] = 768 values --> classification head --> 5 scores --> predicted class

In [9]:
total_params = sum(p.numel() for p in bert.parameters())   # numel() = total number of elements in tensor
print(total_params)     # full parameter dimension of BERT 

109482240


BERT-base
≈ 109 millions of parameters

We want to see the classification head as a Linear layer. 

Until now we loaded only BertModel, so there isn't a way yet to classify towards the 5 labels. We are going to solve this now loading another, more complete, version.

In [10]:
from transformers import AutoModelForSequenceClassification     

# for sequence classification from pretrained :)
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5    # our number of labels 
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


The UNEXPECTED errors are normal because the original parts of pre-trained BERT as cls.predictions
and cls.seq_relationship are not relevant in our case, while they were useful during pre-tranining.

The MISSING status is important to notice as BERT didn't had a specific classifier for our categories, 
so HF will make a new layer and its weights will be initialized from scratch.

In [11]:
outputs = model(**encoded)

print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[-0.0978,  0.1970, -0.0927, -0.3564, -0.0110]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


Logits are the random-initialized raw values of our classification head. 

Loss is None because the model never calculated the loss yet, since we gave **encoded to the model instead of the labels. 

Hidden_state is None too because we never asked BERT to return its internal layer rappresentations. 

Same thing with attention, because we don't have attention weights yet from layers and attention heads.

We could change these hyperparameters to True already, but it's not time yet. 
